# 🧠 Deep Learning for NLP — Sentiment Analysis
### Hands-On Session | ~60 minutes

---

## What We'll Build
A **movie review sentiment classifier** that reads text and predicts whether a review is **Positive** or **Negative** — trained on the real IMDB dataset.

## Learning Journey

| Step | Topic | Time |
|------|-------|------|
| 1 | Setup & Dataset Exploration | 10 min |
| 2 | Text → Numbers (Preprocessing) | 10 min |
| 3 | Building the Deep Learning Model | 10 min |
| 4 | Training & Evaluation | 15 min |
| 5 | Visualizing & Predicting | 10 min |
| 6 | Exercises | 5 min |

## Key Concepts Covered
- Tokenization & Vocabulary
- Sequence Padding
- **Word Embeddings** (the magic of representing words as vectors)
- Dense Neural Network for classification
- Training loop, loss, accuracy

> **No GPU required** — this runs fast on CPU too!

---
## Step 1 — Setup & Dataset Exploration

In [ ]:
# All libraries are pre-installed on Google Colab — no pip install needed!
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print("✅ All imports successful!")

### 📦 The IMDB Dataset

- **50,000** movie reviews from IMDb
- Each review is labeled: `1` = Positive, `0` = Negative
- Split: 25,000 train / 25,000 test
- Keras provides this dataset **pre-tokenized** (words already converted to integers)

We'll use only the **top 10,000 most frequent words** (vocabulary size).

In [ ]:
VOCAB_SIZE = 10000   # Keep only the 10,000 most common words

# Load dataset — already split into train/test
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print("Dataset loaded!")
print(f"  Training samples   : {len(X_train)}")
print(f"  Test samples       : {len(X_test)}")
print(f"  Label values       : {set(y_train)}  (0=Negative, 1=Positive)")

In [ ]:
# Let's look at a raw sample — it's a list of integers (word indices)
sample_idx = 0
print(f"Label      : {'Positive 😊' if y_train[sample_idx] == 1 else 'Negative 😞'}")
print(f"Review     : {X_train[sample_idx][:20]} ...")
print(f"Length     : {len(X_train[sample_idx])} words")

In [ ]:
# 🔍 Decode integers back to actual words
word_index = imdb.get_word_index()
# Reverse: index → word (offset by 3 because 0,1,2 are reserved)
reverse_word_index = {v + 3: k for k, v in word_index.items()}
reverse_word_index[0] = '<PAD>'
reverse_word_index[1] = '<START>'
reverse_word_index[2] = '<UNK>'

def decode_review(encoded_review):
    return ' '.join(reverse_word_index.get(i, '<UNK>') for i in encoded_review)

print("Decoded review:")
print("-" * 60)
print(decode_review(X_train[sample_idx]))
print("-" * 60)
print(f"Label: {'Positive 😊' if y_train[sample_idx] == 1 else 'Negative 😞'}")

In [ ]:
# How long are the reviews?
lengths = [len(r) for r in X_train]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Review length distribution
axes[0].hist(lengths, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(np.median(lengths), color='red', linestyle='--', label=f'Median: {int(np.median(lengths))}')
axes[0].set_title('Review Length Distribution')
axes[0].set_xlabel('Number of words')
axes[0].set_ylabel('Count')
axes[0].legend()

# Class balance
unique, counts = np.unique(y_train, return_counts=True)
axes[1].bar(['Negative', 'Positive'], counts, color=['tomato', 'mediumseagreen'], edgecolor='white')
axes[1].set_title('Class Balance (Training Set)')
axes[1].set_ylabel('Count')
for i, c in enumerate(counts):
    axes[1].text(i, c + 100, str(c), ha='center', fontweight='bold')

plt.suptitle('IMDB Dataset Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Min length: {min(lengths)}, Max length: {max(lengths)}, Median: {int(np.median(lengths))}")

---
## Step 2 — Text Preprocessing: Making Reviews the Same Length

### The Problem
Neural networks expect **fixed-size inputs**, but reviews have different lengths.

### The Solution: Padding
We fix a `MAX_LEN` and either:
- **Pad** short reviews with zeros on the left → `[0, 0, 0, 24, 17, 5]`
- **Truncate** long reviews to keep only the last `MAX_LEN` words

```
Short review : [24, 17, 5]          → [0, 0, 0, 24, 17, 5]   (padded)
Long review  : [1, 2, 3, ..., 999]  → [last 200 words only]   (truncated)
```

In [ ]:
MAX_LEN = 200   # We'll use the last 200 words of each review

X_train_padded = pad_sequences(X_train, maxlen=MAX_LEN, padding='pre', truncating='pre')
X_test_padded  = pad_sequences(X_test,  maxlen=MAX_LEN, padding='pre', truncating='pre')

print(f"Shape before padding : variable length")
print(f"Shape after padding  : {X_train_padded.shape}")
print(f"\nEvery review is now exactly {MAX_LEN} integers long.")

# Show padding in action
print(f"\nFirst review (padded):")
print(X_train_padded[0][:20], '...')

---
## Step 3 — Building the Model

### 🔑 Key Concept: Word Embeddings

Instead of treating words as arbitrary numbers, an **Embedding layer** learns to represent each word as a **dense vector** of real numbers.

```
"king"   →  [0.2,  0.8, -0.1,  0.5,  ...]   (16 numbers)
"queen"  →  [0.2,  0.8, -0.1,  0.4,  ...]   (similar!)
"pizza"  →  [-0.9, 0.1,  0.7, -0.3,  ...]   (very different)
```

Similar words end up with **similar vectors** — the model learns this automatically!

### Our Architecture

```
Input      [200 integers]          ← padded review
    ↓
Embedding  [200 × 16 floats]       ← word → vector lookup
    ↓
GlobalAvgPool  [16 floats]         ← average all word vectors
    ↓
Dense(64, ReLU)  [64 floats]       ← learn patterns
    ↓
Dropout(0.3)                       ← prevent overfitting
    ↓
Dense(1, Sigmoid)  [1 float]       ← 0.0=Negative, 1.0=Positive
```

In [ ]:
EMBEDDING_DIM = 16   # Each word is represented as a 16-dimensional vector

model = keras.Sequential([
    # Layer 1: Embedding — learns word representations
    layers.Embedding(
        input_dim=VOCAB_SIZE,       # vocabulary size (10,000 words)
        output_dim=EMBEDDING_DIM,   # vector size per word
        input_length=MAX_LEN,       # sequence length
        name='embedding'
    ),

    # Layer 2: GlobalAveragePooling — collapses 200 word vectors into one
    layers.GlobalAveragePooling1D(name='avg_pooling'),

    # Layer 3: Dense — learns higher-level patterns
    layers.Dense(64, activation='relu', name='hidden'),

    # Layer 4: Dropout — randomly zeros out 30% of neurons during training
    layers.Dropout(0.3, name='dropout'),

    # Layer 5: Output — single neuron with sigmoid → probability (0 to 1)
    layers.Dense(1, activation='sigmoid', name='output')
], name='sentiment_classifier')

model.summary()

### Compiling the Model

| Setting | Choice | Why |
|---------|--------|-----|
| **Optimizer** | Adam | Adaptive learning rate, works well out of the box |
| **Loss** | Binary Crossentropy | Standard for binary (0/1) classification |
| **Metric** | Accuracy | Easy to understand: % of correct predictions |

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Build explicitly so count_params() works before the first forward pass
model.build(input_shape=(None, MAX_LEN))

print("Model compiled ✅")
print(f"Total trainable parameters: {model.count_params():,}")

---
## Step 4 — Training the Model

We hold out **20% of training data as validation** so we can monitor overfitting in real time.

> **Watch for**: if `val_accuracy` stops improving while `accuracy` keeps rising → **overfitting**!

In [ ]:
EPOCHS     = 10
BATCH_SIZE = 512

history = model.fit(
    X_train_padded, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,   # 20% of training data → validation
    verbose=1
)

print("\n✅ Training complete!")

In [ ]:
# Evaluate on the held-out test set (data the model has never seen)
test_loss, test_acc = model.evaluate(X_test_padded, y_test, verbose=0)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}  ({test_acc*100:.1f}%)")

---
## Step 5 — Visualizing Training & Making Predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs_range = range(1, EPOCHS + 1)

# --- Accuracy plot ---
axes[0].plot(epochs_range, history.history['accuracy'],     'o-', color='steelblue',  label='Train Accuracy')
axes[0].plot(epochs_range, history.history['val_accuracy'], 's--', color='tomato',    label='Val Accuracy')
axes[0].axhline(test_acc, color='green', linestyle=':', linewidth=2, label=f'Test Accuracy ({test_acc:.3f})')
axes[0].set_title('Accuracy over Epochs', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].set_ylim([0.5, 1.0])
axes[0].grid(alpha=0.3)

# --- Loss plot ---
axes[1].plot(epochs_range, history.history['loss'],     'o-', color='steelblue',  label='Train Loss')
axes[1].plot(epochs_range, history.history['val_loss'], 's--', color='tomato',    label='Val Loss')
axes[1].set_title('Loss over Epochs', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 🎯 Predict Sentiment on Custom Text

Now let's test our model on brand-new movie reviews!

In [ ]:
def preprocess_custom_text(text, word_index, max_len=MAX_LEN):
    """Convert a raw text string into a padded integer sequence."""
    # Lowercase and split into words
    words = text.lower().split()
    # Convert words to indices (unknown words → 2 = <UNK>)
    # Note: offset by 3 to match the dataset's indexing
    encoded = [word_index.get(w, 2) + 3 for w in words]
    # Clip to vocab size
    encoded = [min(idx, VOCAB_SIZE - 1) for idx in encoded]
    # Pad to fixed length
    padded = pad_sequences([encoded], maxlen=max_len, padding='pre', truncating='pre')
    return padded


def predict_sentiment(review_text):
    """Predict sentiment for a given review string."""
    processed = preprocess_custom_text(review_text, word_index)
    score = model.predict(processed, verbose=0)[0][0]
    label = 'Positive 😊' if score >= 0.5 else 'Negative 😞'
    confidence = score if score >= 0.5 else 1 - score
    print(f"Review     : {review_text[:80]}..." if len(review_text) > 80 else f"Review     : {review_text}")
    print(f"Prediction : {label}")
    print(f"Confidence : {confidence*100:.1f}%  (raw score: {score:.4f})")
    print()


print("Prediction function ready ✅")

In [ ]:
print("=" * 60)
print("CUSTOM REVIEW PREDICTIONS")
print("=" * 60)
print()

reviews = [
    "This movie was absolutely wonderful! The acting was superb and the story kept me on the edge of my seat the entire time.",
    "Terrible film. Boring plot, awful acting, complete waste of time. I want my two hours back.",
    "It was okay, not great but not terrible either. Some parts were good, some were slow.",
    "One of the best films I have ever seen. A masterpiece of storytelling and emotion.",
]

for review in reviews:
    predict_sentiment(review)
    print("-" * 60)

### 🔬 Bonus: Peek Inside the Embedding Layer

The model learned a **16-dimensional vector** for every word in our vocabulary. Let's see which words ended up close to "good" and "bad".

In [ ]:
# Extract the learned embedding weights
embedding_weights = model.get_layer('embedding').get_weights()[0]
print(f"Embedding matrix shape: {embedding_weights.shape}")
print(f"  → {embedding_weights.shape[0]:,} words × {embedding_weights.shape[1]} dimensions")

In [ ]:
def find_similar_words(target_word, top_n=10):
    """Find words with the most similar embeddings using cosine similarity."""
    if target_word not in word_index:
        print(f"'{target_word}' not in vocabulary.")
        return

    target_idx = word_index[target_word] + 3
    if target_idx >= VOCAB_SIZE:
        print(f"'{target_word}' is outside our vocabulary limit.")
        return

    target_vec = embedding_weights[target_idx]

    # Cosine similarity with all words
    norms = np.linalg.norm(embedding_weights, axis=1, keepdims=True) + 1e-8
    normalized = embedding_weights / norms
    target_norm = target_vec / (np.linalg.norm(target_vec) + 1e-8)
    similarities = normalized @ target_norm

    # Top similar (skip index 0 which is padding)
    top_indices = np.argsort(similarities)[::-1][1:top_n+1]
    print(f"Words most similar to '{target_word}':")
    for rank, idx in enumerate(top_indices, 1):
        word = reverse_word_index.get(idx, '<UNK>')
        print(f"  {rank:2}. {word:<20} (similarity: {similarities[idx]:.3f})")
    print()

find_similar_words('good')
find_similar_words('bad')

---
## Step 6 — Exercises 🏋️

Try these modifications to deepen your understanding:

### Exercise 1 — Change the Embedding Dimension

Try `EMBEDDING_DIM = 32` or `EMBEDDING_DIM = 8`.  
**Question**: Does a bigger embedding always give better accuracy? Why or why not?

**Hint**: Larger embeddings = more expressive, but also more parameters and risk of overfitting.

In [ ]:
# ✏️ YOUR CODE HERE
# Try rebuilding the model with a different EMBEDDING_DIM
# Then compile, fit, and evaluate

NEW_EMBEDDING_DIM = 32  # ← change this!

model_ex1 = keras.Sequential([
    layers.Embedding(VOCAB_SIZE, NEW_EMBEDDING_DIM, input_length=MAX_LEN),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model_ex1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_ex1.fit(X_train_padded, y_train, epochs=5, batch_size=512, validation_split=0.2, verbose=1)
loss, acc = model_ex1.evaluate(X_test_padded, y_test, verbose=0)
print(f"\nEmbedding dim {NEW_EMBEDDING_DIM} → Test Accuracy: {acc*100:.1f}%")

### Exercise 2 — Add a Second Dense Layer

Add another hidden layer between the existing Dense(64) and the output.  
**Question**: Does more depth help here? Can you observe overfitting?

```python
# Add after Dense(64):
layers.Dense(32, activation='relu'),
```

In [ ]:
# ✏️ YOUR CODE HERE


### Exercise 3 — Test Your Own Reviews

Write 3 movie reviews of your own and see what the model predicts.  
Try to **trick** the model with a sarcastic or ambiguous review!

In [ ]:
# ✏️ YOUR CODE HERE
my_reviews = [
    "Write your first review here.",
    "Write your second review here.",
    "Oh wow, what a GREAT waste of my evening...",  # sarcastic — will it fool the model?
]

for review in my_reviews:
    predict_sentiment(review)

### Exercise 4 (Stretch) — Plot Confusion Matrix

Plot a confusion matrix for the test set to see where the model makes mistakes.

```python
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
y_pred = (model.predict(X_test_padded) >= 0.5).astype(int).flatten()
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive']).plot()
plt.show()
```

In [ ]:
# ✏️ YOUR CODE HERE


---
## 📝 Summary — What We Did

| Step | What we did | Key concept |
|------|-------------|-------------|
| 1 | Loaded the IMDB dataset | Real-world NLP data |
| 2 | Decoded integer sequences back to text | Tokenization |
| 3 | Padded sequences to fixed length | Uniform input shape |
| 4 | Built Embedding + Dense model | Word embeddings |
| 5 | Trained for 10 epochs | Gradient descent, backprop |
| 6 | Achieved ~87–88% accuracy | Model evaluation |
| 7 | Predicted on custom reviews | Inference |
| 8 | Explored learned word vectors | Embedding semantics |

## What's Next? 🚀

- **LSTM / GRU** — capture word *order* (not just average meaning)
- **1D CNN** — fast pattern matching over local word windows  
- **Pre-trained embeddings** — GloVe, Word2Vec (better than training from scratch)
- **Transformers / BERT** — state-of-the-art; understands context bidirectionally

---
*Built with TensorFlow/Keras · IMDB dataset · Google Colab compatible*